# MC Dropout -- uncapped RUL

Same method as `dropout_FIXED.ipynb`, trained and evaluated entirely on uncapped RUL. This is the headline abstention result -- the scenario where the model genuinely doesn't know how much life is left early in an engine's run is preserved here, unlike the capped version where that ambiguity is resolved by the label itself.

In [3]:
import pandas as pd
import numpy as np


In [4]:
df = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\processed_train_uncapped.csv")

np.random.seed(42)

engine_ids = df["Engine_ID"].unique()
np.random.shuffle(engine_ids)

train_ids = engine_ids[:70]
val_ids = engine_ids[70:85]
test_ids = engine_ids[85:]

train_df = df[df["Engine_ID"].isin(train_ids)].copy()
val_df = df[df["Engine_ID"].isin(val_ids)].copy()
test_df = df[df["Engine_ID"].isin(test_ids)].copy()

print("Training engines:", train_df["Engine_ID"].nunique())
print("Validation engines:", val_df["Engine_ID"].nunique())
print("Testing engines:", test_df["Engine_ID"].nunique())


Training engines: 70
Validation engines: 15
Testing engines: 15


In [5]:
FEATURES = [
    "Setting_1",
    "Setting_2",
    "Sensor_2",
    "Sensor_3",
    "Sensor_4",
    "Sensor_6",
    "Sensor_7",
    "Sensor_8",
    "Sensor_9",
    "Sensor_11",
    "Sensor_12",
    "Sensor_13",
    "Sensor_14",
    "Sensor_15",
    "Sensor_17",
    "Sensor_20",
    "Sensor_21"
]

TARGET = "RUL"


In [6]:
WINDOW = 30

def create_sequences(data, features, target, window=30):
    X = []
    y = []

    for engine_id, engine_data in data.groupby("Engine_ID"):

        engine_data = engine_data.sort_values("Cycle")

        feature_values = engine_data[features].values.astype("float32")
        target_values = engine_data[target].values.astype("float32")

        for i in range(window, len(engine_data)):
            X.append(feature_values[i-window:i])
            y.append(target_values[i])

    return np.array(X), np.array(y)


In [7]:
X_train, y_train = create_sequences(train_df, FEATURES, TARGET, WINDOW)
X_val, y_val = create_sequences(val_df, FEATURES, TARGET, WINDOW)
X_test, y_test = create_sequences(test_df, FEATURES, TARGET, WINDOW)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)


(12216, 30, 17)
(2720, 30, 17)
(2695, 30, 17)


## BiLSTM

In [8]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Bidirectional,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.random.set_seed(42)

bilstm = Sequential([

    Input(shape=(WINDOW, len(FEATURES))),

    Bidirectional(
        LSTM(
            64,
            return_sequences=True
        )
    ),

    Dropout(0.2),

    Bidirectional(
        LSTM(32)
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

bilstm.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

bilstm.summary()


C:\Users\Shivam\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.2) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 30, 128)        │        41,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,313 (333.25 KB)

 Trainable params: 85,313 (333.25 KB)

 Non-trainable params: 0 (0.00 B)

In [9]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

history_bilstm = bilstm.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,
    batch_size=64,

    shuffle=False,

    callbacks=[
        early_stopping,
        reduce_lr
    ],

    verbose=1
)


Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 11s 30ms/step - loss: 7181.9199 - mae: 66.6788 - val_loss: 3881.7861 - val_mae: 50.5575 - learning_rate: 0.0010
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 3835.4751 - mae: 50.1179 - val_loss: 3651.9294 - val_mae: 50.1639 - learning_rate: 0.0010
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 3841.7368 - mae: 50.4364 - val_loss: 3654.0342 - val_mae: 50.2001 - learning_rate: 0.0010
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 3832.6738 - mae: 50.3771 - val_loss: 3654.3538 - val_mae: 50.2060 - learning_rate: 0.0010
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 3831.3147 - mae: 50.3277 - val_loss: 3653.5964 - val_mae: 50.1955 - learning_rate: 0.0010
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - loss: 3825.0012 - mae: 50.3090 - val_loss: 3239.1045 - val_mae: 47.1016 - learning_rate: 0.0010
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 2431.6658 - mae: 36.6203 

In [10]:
def mc_dropout_predict(model, X, n_passes=100):

    predictions = []

    for _ in range(n_passes):

        pred = model(
            X,
            training=True
        ).numpy().ravel()

        predictions.append(pred)

    predictions = np.array(predictions)

    mean_pred = predictions.mean(axis=0)
    std_pred = predictions.std(axis=0)

    return mean_pred, std_pred


In [11]:
mc_mean, mc_std = mc_dropout_predict(
    bilstm,
    X_test,
    n_passes=100
)

print("Predictions:", mc_mean.shape)
print("Uncertainty:", mc_std.shape)

print("\nMean uncertainty:", mc_std.mean())
print("Min uncertainty :", mc_std.min())
print("Max uncertainty :", mc_std.max())


Predictions: (2695,)
Uncertainty: (2695,)

Mean uncertainty: 8.885308
Min uncertainty : 2.903421
Max uncertainty : 13.73987


In [12]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Held-out split:")
print("MAE :", mean_absolute_error(y_test, mc_mean))
print("RMSE:", np.sqrt(mean_squared_error(y_test, mc_mean)))
print("R\u00b2  :", r2_score(y_test, mc_mean))


Held-out split:
MAE : 32.61708450317383
RMSE: 42.88431761455025
R²  : 0.5506532788276672


## Calibrate the accept/abstain threshold on `val_df`

In [13]:
mc_cal_mean, mc_cal_std = mc_dropout_predict(
    bilstm,
    X_val,
    n_passes=100
)

ACCEPT_RATE = 0.80

mc_threshold = np.quantile(
    mc_cal_std,
    ACCEPT_RATE
)

print("MC uncertainty threshold:", mc_threshold)


MC uncertainty threshold: 10.672575


## Official test set (uncapped)

In [14]:
test_df_final = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\processed_test.csv")
output_df = pd.read_csv(
    r"C:\Users\Shivam\Desktop\Capstone_updated\RUL_FD001.txt",
    header=None,
    names=["RUL"]
)
output_df["Engine_ID"] = np.arange(1, len(output_df) + 1)
# NOTE: ground truth left uncapped on purpose -- this notebook trains on
# uncapped RUL, so evaluation has to stay uncapped too, or predictions and
# ground truth would no longer be answering the same question.

def create_test_sequences(test_df, features, window=30):
    X, engine_ids = [], []
    for engine_id, engine_data in test_df.groupby("Engine_ID"):
        engine_data = engine_data.sort_values("Cycle")
        values = engine_data[features].values.astype("float32")
        if len(values) >= window:
            X.append(values[-window:])
            engine_ids.append(engine_id)
    return np.array(X), np.array(engine_ids)

X_test_final, test_engine_ids = create_test_sequences(test_df_final, FEATURES, WINDOW)


In [15]:
mc_test_mean, mc_test_std = mc_dropout_predict(bilstm, X_test_final, n_passes=100)

pred_df = pd.DataFrame({"Engine_ID": test_engine_ids, "Predicted_RUL": mc_test_mean, "Uncertainty": mc_test_std})
comparison = pred_df.merge(output_df, on="Engine_ID")
actual, mc_test_mean, mc_test_std = comparison["RUL"].values, comparison["Predicted_RUL"].values, comparison["Uncertainty"].values


In [16]:
def nasa_score(actual, predicted):
    """NASA / PHM08 asymmetric RUL scoring function (Saxena et al., 2008).
    Penalizes late predictions (predicted RUL > actual RUL) more heavily
    than early ones, since a late prediction means maintenance would be
    scheduled after the engine could already have failed.
    """
    d = np.asarray(predicted) - np.asarray(actual)
    s = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return np.sum(s)


In [17]:
print("100% coverage:")
print("MAE  :", mean_absolute_error(actual, mc_test_mean))
print("RMSE :", np.sqrt(mean_squared_error(actual, mc_test_mean)))
print("R\u00b2   :", r2_score(actual, mc_test_mean))
print("Score:", nasa_score(actual, mc_test_mean))


100% coverage:
MAE  : 25.513427734375
RMSE : 31.806547975859075
R²   : 0.4141674041748047
Score: 11776.14862878464


## Accept / abstain sweep

In [18]:
accept = mc_test_std <= mc_threshold

print("Accepted :", accept.sum())
print("Abstained:", (~accept).sum())
print("Coverage :", accept.mean())


Accepted : 82
Abstained: 18
Coverage : 0.82


In [19]:
y_true_accept = actual[accept]
y_pred_accept = mc_test_mean[accept]

mae = mean_absolute_error(y_true_accept, y_pred_accept)
rmse = np.sqrt(mean_squared_error(y_true_accept, y_pred_accept))
r2 = r2_score(y_true_accept, y_pred_accept)
score = nasa_score(y_true_accept, y_pred_accept)

print("Accepted predictions:")
print("MAE  :", mae)
print("RMSE :", rmse)
print("R\u00b2   :", r2)
print("Score:", score)


Accepted predictions:
MAE  : 23.422969818115234
RMSE : 29.29159431575198
R²   : 0.48512452840805054
Score: 7345.45740723338


In [20]:
for rate in [0.9, 0.8, 0.7]:

    threshold = np.quantile(mc_cal_std, rate)
    accept = mc_test_std <= threshold

    rmse = np.sqrt(mean_squared_error(actual[accept], mc_test_mean[accept]))
    score = nasa_score(actual[accept], mc_test_mean[accept])

    print(
        f"Acceptance={rate:.0%} | "
        f"Accepted={accept.sum()} | "
        f"RMSE={rmse:.3f} | "
        f"Score={score:.1f}"
    )


Acceptance=90% | Accepted=92 | RMSE=29.894 | Score=9654.4
Acceptance=80% | Accepted=82 | RMSE=29.292 | Score=7345.5
Acceptance=70% | Accepted=76 | RMSE=29.121 | Score=7073.4


In [21]:
import os

METHOD = "mc_dropout"
SCENARIO = "clean_uncapped"

os.makedirs("results", exist_ok=True)

predicted = mc_test_mean
uncertainty = mc_test_std

results_df = pd.DataFrame({
    "Engine_ID": test_engine_ids,
    "Actual_RUL": actual,
    "Predicted_RUL": predicted,
    "Uncertainty": uncertainty
})

out_path = f"results/{METHOD}__{SCENARIO}.csv"
results_df.to_csv(out_path, index=False)

print(f"Saved {len(results_df)} rows -> {out_path}")

Saved 100 rows -> results/mc_dropout__clean_uncapped.csv


In [22]:
print(results_df.head())
print()
print("Shape:", results_df.shape)
print()
print("Missing values:")
print(results_df.isna().sum())

   Engine_ID  Actual_RUL  Predicted_RUL  Uncertainty
0          1         112     148.417557    10.864874
1          2          98     136.879135     9.227784
2          3          69      84.007370     8.346481
3          4          82     100.430733     8.535007
4          5          91     118.073532     9.279219

Shape: (100, 4)

Missing values:
Engine_ID        0
Actual_RUL       0
Predicted_RUL    0
Uncertainty      0
dtype: int64
